# Lighter-Weight Fact Verification Approach -> Sentence-Level (No NER)

This notebook evaluates sentence-level candidates for fact verification without using a Named Entity Recognition (NER) pre-filter.

What this notebook does:

1. Creates the sentences and loads the annotated data
2. Examines the exact-match results
3. Examines String-based comparison results 
4. Examines Embedding similarity results 
5. Examines Paraphrase detection results 

All helper functions are placed inside `fact_check_utils.py` (imported in this notebook as `fc`).

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import fact_check_utils as fc

## Paths

Here the paths to the data are mentioned. One path leads to the JSON file, and the other to the text extracted from the award documents.

In [ ]:
json_file_path = r"...\annotated_facts_2026-03-31.json"
file_path      = r"...\text_test"

## 1. Load annotated data

In [ ]:
full_data_json = fc.read_json(json_file_path) 
info_results_json = fc.json_feature(full_data_json) # capturing the full information from JSON

all_facts   = fc.build_all_facts(info_results_json)
gold_labels = fc.build_gold_labels(info_results_json)   # shared by every method: 1=verified, 0=contradicted
print(f"{len(all_facts)} facts present")

## 2. Sentence reading (No NER pre-filter)

In [ ]:
sent_cache = {}
for info in info_results_json:
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in sent_cache and os.path.exists(actual_path):
        sentences = fc.sent_sent_reading(actual_path)
        sent_cache[actual_path] = sentences
        print(f"  Cached: {actual_path} ({len(sentences)} sentences)")
print(f"Total files cached: {len(sent_cache)}")

## 3. Exact Match Baseline

In [ ]:
# fc.exact_match_score = substring exact match (1 if fact appears verbatim in a sentence)
exact_scores = []
gold_labels = gold_labels 

for info in info_results_json:
    facts = info["fact"]
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in sent_cache:
        print("WRONG FILE")
        continue

    sentences = sent_cache[actual_path]
    score = fc.exact_match_score(facts, sentences)
    exact_scores.append(score)

    if score == 1:
        print(f"MATCH FOUND in {actual_path}")
        print(f"  Fact: {facts}")
        # Find which sentence matched
        for sent in sentences:
            if facts.strip().lower() in sent.strip().lower():
                print(f"  Sentence: {sent}")
                break
    print()

print(f"\nTotal matches: {sum(exact_scores)} out of {len(exact_scores)}")
print(classification_report(gold_labels, exact_scores, digits=3, target_names=["Contradicted", "Verified"]))

## 4. String-Based —> Levenshtein

In [ ]:
lev_chosen_thresh = [0.0, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1]

for thresh in lev_chosen_thresh:
    predicted_lev = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in sent_cache:
            print("WRONG FILE")

        label_lev, score_lev, sentence_lev = fc.get_best_match_ner_lev(
            info["fact"], sent_cache[actual_path], "og_ratio", thresh
        )
        predicted_lev.append(label_lev)

    fc.report_at_threshold(gold_labels, predicted_lev, thresh)

## 5. String-Based —> RapidFuzz

In [ ]:
rf_chosen_thresh = [0, 50, 55, 60, 65, 70, 75, 80, 85, 90, 100]

for thresh in rf_chosen_thresh:
    predicted_rf = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in sent_cache:
            print("WRONG FILE")

        label_rf, score_rf, sentence_rf = fc.get_best_match_ner_rapfuz(
            info["fact"], sent_cache[actual_path], "token_sort", thresh
        )
        predicted_rf.append(label_rf)
    
    fc.report_at_threshold(gold_labels, predicted_rf, thresh)

## 6. Embedding Similarity -> all-MiniLM-L6-v2

In [ ]:
# model used for Embedding Similarity Approach
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# building the embedding cache for the sentences using build_embedding_cache and save it.
# Then just load the pickle below.
embedding_cache_emb = fc.build_embedding_cache(sent_cache, model, show_progress_bar=True)
with open("new_embedding_cache_emb_no_ner.pkl", "wb") as f:
    pickle.dump(embedding_cache_emb, f)

In [ ]:
with open("new_embedding_cache_emb_no_ner.pkl", "rb") as emb:
    embedding_cache_emb = pickle.load(emb)

In [ ]:
chosen_thresh_emb = [0.0, 0.2, 0.5, 0.55, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.8, 0.85, 0.9, 1]

for thresh in chosen_thresh_emb:
    emb_predicted = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in sent_cache:
            print("WRONG FILE")

        score_emb, matched_sentence_emb, found_emb = fc.get_first_above_thresh(
            info["fact"], sent_cache[actual_path],
            embedding_cache_emb[actual_path], model, thresh
        )
        emb_predicted.append(1 if found_emb else 0)

    fc.report_at_threshold(gold_labels, emb_predicted, thresh)

## 7. Paraphrase Detection —> paraphrase-mpnet-base-v2

In [ ]:
# model used for Paraphrase Detection Approach
model_paraph = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")

In [ ]:
# building the embedding cache for the sentences using build_embedding_cache and save it.
# Then just load the pickle below.
embedding_cache_paraph = fc.build_embedding_cache(sent_cache, model_paraph, show_progress_bar=True)
with open("new_embedding_cache_paraph_no_ner.pkl", "wb") as f:
     pickle.dump(embedding_cache_paraph, f)

In [ ]:
with open("new_embedding_cache_paraph_no_ner.pkl", "rb") as paraph:
    embedding_cache_paraph = pickle.load(paraph)

In [ ]:
chosen_thresh_paraph = [0.0, 0.2, 0.5, 0.55, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.77, 0.8, 0.82, 0.85, 0.9, 1]

for thresh in chosen_thresh_paraph:
    paraph_predicted = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in sent_cache:
            print("WRONG FILE")

        score_paraph, matched_sentence_paraph, found_paraph = fc.get_first_above_thresh_paraph(
            info["fact"], sent_cache[actual_path],
            embedding_cache_paraph[actual_path], model_paraph, thresh
        )
        paraph_predicted.append(1 if found_paraph else 0)

    fc.report_at_threshold(gold_labels, paraph_predicted, thresh)